## Step 0: Mount Google Drive
This cell establishes a connection between Google Colab and your Google Drive. This is essential for accessing the standardized images provided by Role 2 and for saving our Week 2 experimental results in a persistent way.

In [1]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the project root directory (aligned with Week 1 for continuity)
PROJECT_ROOT = '/content/drive/MyDrive/ML2-Team2/Final Project/face-to-bmi'

if os.path.exists(PROJECT_ROOT):
    print(f"✅ Project folder linked: {PROJECT_ROOT}")
else:
    print(f"❌ Project folder NOT found. Please verify the path in your Drive.")

Mounted at /content/drive
✅ Project folder linked: /content/drive/MyDrive/ML2-Team2/Final Project/face-to-bmi


## Cell 1: Week 2 Environment Setup and Role 2 Synchronization
In Week 2, we shift our focus to the **standardized** dataset. We update our `DATA_DIR` to point to `faces_standardized/`, which uses padding to preserve facial proportions—a key factor for BMI prediction. We also link to the `clean_data.csv` generated in Week 1 to ensure we are working with the verified subset of 3,962 images. All outputs for this week will be stored in the `week2_experiments` directory.

In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch

# --- Global Reproducibility ---
# We maintain SEED 42 to ensure our Week 2 results are directly comparable to the Week 1 baseline.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Path Definitions (Updated for Week 2) ---
# 1. New Source: Role 2's Recommended Version 2 (Standardized)
DATA_DIR = os.path.join(PROJECT_ROOT, "data/processed/faces_standardized")

# 2. Metadata: The Cleaned list from Week 1
CSV_PATH = os.path.join(PROJECT_ROOT, "data/processed/clean_data.csv")

# 3. Output: Dedicated Week 2 Experiment folder
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "models/ryan/week2_experiments")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Configuration ---
IMG_SIZE = 224
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"--- Week 2 Configuration Loaded ---")
print(f"Execution Device: {DEVICE}")
print(f"Image Source:     {DATA_DIR}")
print(f"Metadata Source:  {CSV_PATH}")

# Path Integrity Check
if os.path.exists(DATA_DIR) and os.path.exists(CSV_PATH):
    print("✅ All Week 2 data sources verified and ready.")
else:
    print("❌ Warning: Path mismatch. Please ensure 'clean_data.csv' exists in 'data/processed/'.")

--- Week 2 Configuration Loaded ---
Execution Device: cuda
Image Source:     /content/drive/MyDrive/ML2-Team2/Final Project/face-to-bmi/data/processed/faces_standardized
Metadata Source:  /content/drive/MyDrive/ML2-Team2/Final Project/face-to-bmi/data/processed/clean_data.csv
✅ All Week 2 data sources verified and ready.


## Cell 2: Dataset and DataLoader Implementation
This cell implements the `FaceBMIDataset` class optimized for our standardized inputs. Since Role 2 has already handled the aspect-ratio-preserving padding and resizing, our transformation pipeline focuses on tensor conversion and ImageNet normalization. We leverage the `is_training` flag to maintain a consistent split for Role 4's upcoming regression tests.

In [3]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class FaceBMIDataset(Dataset):
    def __init__(self, csv_file, root_dir, is_train=True, transform=None):
        self.metadata = pd.read_csv(csv_file)
        # Filter based on train/test split
        target_flag = 1 if is_train else 0
        self.metadata = self.metadata[self.metadata['is_training'] == target_flag].reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.metadata.loc[idx, 'name'])
        try:
            # Role 2 noted 8 images are in 'P' mode; convert('RGB') ensures consistency.
            image = Image.open(img_name).convert('RGB')
        except Exception as e:
            print(f"Warning: Skipping {img_name} due to error: {e}")
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE))

        bmi = float(self.metadata.loc[idx, 'bmi'])
        if self.transform:
            image = self.transform(image)

        return {
            'image': image,
            'bmi': torch.tensor(bmi, dtype=torch.float32),
            'filename': self.metadata.loc[idx, 'name']
        }

# VGG-Face Spec Transformation
vgg_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Instantiate DataLoaders
train_dataset = FaceBMIDataset(CSV_PATH, DATA_DIR, is_train=True, transform=vgg_transform)
test_dataset = FaceBMIDataset(CSV_PATH, DATA_DIR, is_train=False, transform=vgg_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"✅ DataLoaders initialized for standardized data.")
print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples:  {len(test_dataset)}")

✅ DataLoaders initialized for standardized data.
Training samples: 3210
Testing samples:  752


## Cell 3: VGG-Face Model Architecture and Truncation (fc6)
In this cell, we initialize the VGG16 architecture and truncate it to the **fc6** layer. Although we are using the same layer as Week 1, this extraction is performed on images that have been processed to preserve aspect ratios (Standardized), which should provide more stable geometric features for BMI prediction. We set the model to `eval()` mode and ensure all parameters are frozen to act as a deterministic feature extractor.

In [4]:
# Cell 3: VGG-Face Model Architecture and Truncation (fc6)
import torchvision.models as models
import torch.nn as nn

print("Loading VGG16 for standardized feature extraction...")

# Load VGG16 with ImageNet weights
vgg_model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

# --- Truncate to fc6 ---
# Index 0: Linear (fc6), Index 1: ReLU
truncated_classifier = nn.Sequential(*list(vgg_model.classifier.children())[:2])
vgg_model.classifier = truncated_classifier

# Freeze weights
for param in vgg_model.parameters():
    param.requires_grad = False

# Deploy to GPU
vgg_model = vgg_model.to(DEVICE)
vgg_model.eval()

print("✅ fc6 Extractor initialized and pushed to GPU.")
print(f"Target dimension: {vgg_model.classifier[0].out_features}")

Loading VGG16 for standardized feature extraction...
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 170MB/s]


✅ fc6 Extractor initialized and pushed to GPU.
Target dimension: 4096


## Cell 4: Feature Extraction Execution (v2_fc6)
We now execute the extraction loop. The 4096-dimensional features are aggregated into NumPy arrays. To maintain a clean experimental history, we save these files with the suffix `_v2_fc6.npy` in our `week2_experiments` directory. This allows Role 4 to easily perform an A/B test between the Week 1 (raw resize) and Week 2 (standardized padding) features.

In [5]:
# Cell 4: Feature Extraction Loop
from tqdm.auto import tqdm

def extract_features_v2(data_loader, model, device):
    features_list, bmi_list, filenames_list = [], [], []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Extracting"):
            images = batch['image'].to(device)
            bmis = batch['bmi'].numpy()
            filenames = batch['filename']

            # Forward pass
            batch_features = model(images)

            # Aggregate on CPU
            features_list.append(batch_features.cpu().numpy())
            bmi_list.extend(bmis)
            filenames_list.extend(filenames)

    return np.vstack(features_list), np.array(bmi_list), np.array(filenames_list)

print("🚀 Extracting features from STANDARDIZED training set...")
train_features_v2, train_bmis_v2, train_filenames_v2 = extract_features_v2(train_loader, vgg_model, DEVICE)

print("🚀 Extracting features from STANDARDIZED testing set...")
test_features_v2, test_bmis_v2, test_filenames_v2 = extract_features_v2(test_loader, vgg_model, DEVICE)

# --- Save Week 2 Results ---
print(f"💾 Saving v2 features to: {OUTPUT_DIR}")

# We use specific filenames to avoid confusion with Week 1 baseline
np.save(os.path.join(OUTPUT_DIR, 'train_features_v2_fc6.npy'), train_features_v2)
np.save(os.path.join(OUTPUT_DIR, 'train_bmis_v2.npy'), train_bmis_v2)
np.save(os.path.join(OUTPUT_DIR, 'train_filenames_v2.npy'), train_filenames_v2)

np.save(os.path.join(OUTPUT_DIR, 'test_features_v2_fc6.npy'), test_features_v2)
np.save(os.path.join(OUTPUT_DIR, 'test_bmis_v2.npy'), test_bmis_v2)
np.save(os.path.join(OUTPUT_DIR, 'test_filenames_v2.npy'), test_filenames_v2)

print("✅ Week 2 (fc6) Extraction Completed.")

🚀 Extracting features from STANDARDIZED training set...


Extracting:   0%|          | 0/101 [00:00<?, ?it/s]

🚀 Extracting features from STANDARDIZED testing set...


Extracting:   0%|          | 0/24 [00:00<?, ?it/s]

💾 Saving v2 features to: /content/drive/MyDrive/ML2-Team2/Final Project/face-to-bmi/models/ryan/week2_experiments
✅ Week 2 (fc6) Extraction Completed.
